# Corrigé · Exercice 3.1 : un quatrième spécialiste, Accès

> Notebook autonome. Ce corrigé est aussi intégré au jalon J3 (`helpdesk_ops.jalons.j3`).

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import time

from langchain.agents import create_agent
from langchain_core.tools import tool

from helpdesk_ops import kb
from helpdesk_ops.multi_agents import construire_coordinateur, construire_specialistes

specialistes = construire_specialistes(llm)   # kb, diagnostic, ticketing (J3-02)

# Annuaire simulé de NordTech (identifiants pseudonymisés)
COMPTES = {
    "collab-3fa91c": "actif",
    "collab-8b20d4": "verrouillé (3 échecs de connexion, déverrouillage automatique dans 15 min)",
    "collab-71e0aa": "mot de passe expiré",
    "collab-c45f12": "désactivé (départ le 2026-09-30)",
}

QUESTIONS = [
    "Mon compte collab-8b20d4 est bloqué, que faire ?",                        # → acces
    "Outlook plante dès que j'ouvre une pièce jointe",                          # → kb (PAS acces)
    "Le compte collab-71e0aa ne marche plus, et srv-erp-02 répond-il au ping ?",  # → acces ET diagnostic
]


async def poser(coordinateur, question: str, session: str) -> None:
    debut = time.perf_counter()
    with propagate_attributes(session_id=session, tags=["jour-3", "ex31"]):
        etat = await coordinateur.ainvoke({"messages": [("user", question)]},
                                          config={"callbacks": [langfuse_handler], "run_name": "ex31", "recursion_limit": 25})
    appels = [c["name"] for m in etat["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print(f"⏱ {time.perf_counter() - debut:4.0f} s | {appels}\n❓ {question}\n🤖 {etat['messages'][-1].content[:300]}\n")

## 1. Les outils

Le filtre `categorie="acces"` est posé **dans l'outil** : le spécialiste ne peut pas chercher ailleurs, quoi qu'il demande. Le périmètre se garantit par le code, pas par le prompt.

In [ ]:
@tool(parse_docstring=True)
def etat_compte(identifiant: str) -> str:
    """État du compte NordTech d'un collaborateur dans l'annuaire (actif, verrouillé, expiré, désactivé).

    Args:
        identifiant: identifiant pseudonymisé du collaborateur, par exemple 'collab-8b20d4'
    """
    return COMPTES.get(identifiant.strip().lower(), f"compte inconnu : {identifiant}")


@tool(parse_docstring=True)
def procedures_acces(question: str) -> str:
    """Cherche les procédures du help-desk NordTech sur les comptes, mots de passe, MFA et droits d'accès.

    Args:
        question: le problème d'accès en une phrase, par exemple 'compte verrouillé après plusieurs échecs'
    """
    return kb.formater(kb.rechercher(question, k=3, categorie="acces"))


print(etat_compte.invoke({"identifiant": "collab-8b20d4"}))
print(procedures_acces.invoke({"question": "compte verrouillé"})[:300], "…")

## 2. Le spécialiste et le coordinateur

In [ ]:
specialiste_acces = create_agent(
    llm, [etat_compte, procedures_acces], name="specialiste_acces",
    system_prompt="Tu es le spécialiste Accès du help-desk NordTech. Vérifie l'état du compte avec etat_compte, puis indique "
                  "la procédure à suivre en citant [KB-xxx]. Tu ne réinitialises jamais un mot de passe toi-même.",
)

DESCRIPTION_ACCES = "Vérifie l'état du compte d'un collaborateur (verrouillé, expiré, désactivé) et donne la procédure d'accès."
coordinateur = construire_coordinateur(llm, {**specialistes, "acces": specialiste_acces}, {"acces": DESCRIPTION_ACCES})

## 3. Les tests

In [ ]:
INITIALES = "corrige"
for q in QUESTIONS:
    await poser(coordinateur, q, f"ex31-{INITIALES}")
langfuse.flush()

## Ce qu'il fallait observer

- **Question 1** : `demander_acces` seul ; la réponse dit « verrouillé, déverrouillage dans 15 min » et cite une procédure de la catégorie accès (KB-002 ou KB-013).
- **Question 2** : `demander_kb` seul (KB-004) : la description précise de l'Accès évite qu'il soit appelé « au cas où ».
- **Question 3** : `demander_acces` **et** `demander_diagnostic`, généralement l'un après l'autre (le modèle émet parfois les deux appels dans le même message). Comptez les appels LLM : coordinateur (2 tours) + 2 par spécialiste. Un spécialiste de plus, c'est un appel de décision un peu plus long à **chaque** question, et deux appels de plus quand il est sollicité.
- **Consignes** : le coordinateur reformule (« Vérifier l'état du compte collab-71e0aa… ») ; l'identifiant pseudonymisé passe, le reste de la conversation non.

## Pour aller plus loin

- Remplacez `DESCRIPTION_ACCES` par « gère les accès » et relancez la question 2 : le coordinateur appelle-t-il Accès à tort ?
- Le spécialiste Accès ne doit jamais réinitialiser un mot de passe. Comment le **garantir** (et pas seulement le demander) ? *(ne pas lui donner l'outil : moindre privilège ; au red-team du J4, la liste blanche à l'exécution)*